# Step 1 · Prepare planarian and human inputs

Each step presents the planarian analysis first, followed by the separate human analysis.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT=next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'dataset').is_dir() and (p/'results').is_dir()), None)
assert ROOT is not None, 'Start Jupyter within the repository'
assert (ROOT/'dataset').is_dir() and (ROOT/'results').is_dir(), 'Start Jupyter from the repository root or notebooks/'
sys.path.insert(0,str(ROOT))
PROJECT=ROOT/'dataset'
PANDO=ROOT/'results/pando'
MODEL=ROOT/'results/model'
SCENIC=ROOT/'results/scenic'
SCENIC_DATA=ROOT/'dataset/scenic'
pd.set_option('display.max_columns',30)

print('Package:',ROOT)
print('Python:',sys.version.split()[0])
print('Directories available:', all(p.exists() for p in [PROJECT,PANDO,MODEL,SCENIC]))

## Planarian · input preparation

The primary planarian workflow uses SMED/SMESG identifiers, ATAC PCA and regulatory-network inputs. The large RNA PCA matrix is supplied separately. These data are distinct from the human brain organoid data shown below.


In [ ]:
planarian=ROOT/'dataset/legacy/planarian'
planarian_mapping=pd.read_csv(planarian/'data/smed_smesg_mapping.csv')
planarian_edges=pd.read_csv(planarian/'example_data/edges.csv',index_col=0)
print('Planarian mapped identifiers:',len(planarian_mapping),'input edges:',len(planarian_edges))
print(planarian_mapping.head().to_string(index=False))
print('RNA PCA matrix present:',(planarian/'data/rna_pca_new.csv').exists())


## Human · input preparation

The analysis uses human brain-organoid RNA and ATAC data. RNA and ATAC observations were computationally matched, then aggregated and quality controlled before Pando inference. This notebook verifies the saved inputs and first-stage network. It does not rerun the original Pando preprocessing, whose scripts contain historical absolute paths and require the raw ATAC/fragments in the separate archive.

In [ ]:
for label,p in {'Candidate table':PROJECT/'inputs/PRIORITY_with_strong_local_DA_peak.tsv',
                'Pando network':PANDO/'TF_gene_network.csv',
                'Model weights':MODEL/'model.pt',
                'SCENIC summaries':SCENIC/'full/pruning_summary.json'}.items():
 print(f'{label:20s} {p.relative_to(ROOT)}  exists={p.exists()}')

## First-stage Pando network

Pando models TF expression × chromatin accessibility to obtain TF–region–gene coefficients. Region-level tests are globally BH-adjusted before aggregation to TF–gene edges.

In [ ]:
summary=json.loads((PANDO/'summary.json').read_text())
print(json.dumps(summary,indent=2))
network=pd.read_csv(PANDO/'TF_gene_network.csv')
regional=pd.read_csv(PANDO/'significant_TF_region_gene.tsv.gz',sep='\t')
print('TF–gene rows:',len(network),'regional significant rows:',len(regional))
assert len(network)==49764 and len(regional)==62354
assert regional.padj.lt(.05).all()
print('TFs:',network.tf.nunique(),'target genes:',network.target.nunique())

## GLI3 and highlighted genes

The Pando `padj_min` is the smallest adjusted region-level P value, not a TF–gene joint test. HES4 and PAX6 are present in the initial network; HES5 is not.

In [ ]:
watch=['HES4','HES5','PAX6']
gli=network[(network.tf=='GLI3') & network.target.isin(watch)]
print(gli[['tf','target','estimate','n_regions','padj_min']].to_string(index=False))
assert set(gli.target)=={'HES4','PAX6'}

## Features for the current model

The next step uses training-network in-degree features, RNA correlation, motif similarity and GRNBoost2 edge importance. **GRNBoost2 is therefore an input feature of pando+GRN**, even though its separate baseline comparison is presented in Step 4. The SCENIC preprocessing was not independent of the model input.

In [ ]:
features=pd.read_csv(SCENIC/'adjacencies.tsv',sep='\t')
print('GRNBoost2 feature edges:',len(features))
print('Model features:',json.loads((MODEL/'manifest.json').read_text())['train_features'])

**Reproduction boundary:** `src/legacy/pando_original/METHODS.md` and source snapshots describe input matching, motif resources, GLM and BH correction. The archived R/shell scripts contain `/home/wwd/py_code/mhr/final` paths; running them unchanged can write to their original location. A full original-data rerun is not claimed here.